# nb02 · Chẩn đoán số học: A (DC ngây thơ), B (ổn định của HM), C (gauge centring)

**Mục tiêu.** Nhóm thực nghiệm A của `gauge-centering-xai/paper-idea.md` (§5). Với cùng một mạng $f$ và một điểm $x^\star$, ta dựng một cặp DC $f = P - Q$ bằng ba cách, rồi khôi phục $f(x^\star) = P - Q$ và $\nabla f(x^\star) = \nabla P - \nabla Q$ trong **float64**:

| | Phương pháp | Đầu vào cần có | Siêu tham số |
| --- | --- | --- | --- |
| **A** | cặp NDC đệ quy ngây thơ (Paper1, Bổ đề 1) | trọng số | — |
| **B** | Hidden Monotonicity, chạy đúng như code chính thức: co theo ngưỡng + **ghi đè bằng cache của mạng gốc** + dịch $\alpha$ | trọng số, lượt đi và lượt về của mạng gốc, cache | $\theta = 0.1$, $\Theta = 10$, $\alpha = 0.4$ |
| **C** | gauge centring (Paper1, Mệnh đề 17) | lượt đi + autograd | $C$ (chỉ cần khi xét ngoài tâm) |

Năm mạng, khó dần:
1. **ReLU MLP**: trọng số cố định, mũ đơn.
2. **Mạng cổng**: có nút nhân tự cường.
3. **VGG16**: mạng thật, mũ đơn.
4. **MHA linear**: hai nút song tuyến mỗi khối.
5. **MHA softmax**: không có cách dựng đệ quy.

**Tiêu chí.** Sai số tương đối của giá trị và gradient; số điều kiện $\chi$; độ sâu chứng thực $L^*(\delta)$; chi phí, gồm bộ nhớ phải giữ và thời gian chạy.

**Cách chạy.** Chỉ đổi `N_IMAGES` ở §1:
- `1`: chạy thử. Mọi cell chạy, nhưng MHA chỉ huấn luyện 50 bước và chỉ xét $N \in \{1, 2\}$, nên kết luận chưa có giá trị.
- `5`: chạy đủ.

Từ terminal, trong thư mục `gauge-centering-xai/experiments`, có thanh tiến trình:
```
uv run python scripts/run_notebook.py nb02                  # như trong notebook (chạy thử)
uv run python scripts/run_notebook.py nb02 --n-images 5     # chạy đủ
```
Kết quả nằm ở `results/nb02/n{N_IMAGES}/`. Bản notebook đã chạy là `nb02_numerical_diagnosis.executed.ipynb` trong cùng thư mục đó. Mô hình MHA của lần chạy đủ chỉ huấn luyện một lần rồi lưu ở `models/nb02/`; các lần chạy sau nạp lại. Mô hình 50 bước của lần chạy thử nằm riêng trong thư mục chạy.

**Lưu kết quả, tách tính toán và hiển thị.** Mỗi mục có hai loại cell:
- **cell tính toán**: tính rồi lưu số liệu gốc vào `data/` (CSV, tên cột tiếng Anh cố định như `depth`, `method`, `err_val`);
- **cell hiển thị**: dòng đầu là `# ▶ Hiển thị`; cell này chỉ đọc `data/` rồi vẽ hình hoặc lập bảng qua `scripts/nb02_report.py`.

Tên hiển thị lấy từ `LABELS` (§11). Muốn đổi nhãn thì sửa `LABELS` rồi chạy lại cell hiển thị, không phải tính lại.

**Độc lập.** Notebook chỉ dùng:
- code trong `scripts/`: `dcpair.py` (số học cặp, ba phương pháp, đếm bộ nhớ), `nb02_report.py`, `nb_progress.py`;
- ảnh trong `docs/data/`;
- repo HM chính thức, chỉ để kiểm khớp ở §7.2.

Notebook không đọc gì từ `docs/experiments/`. Phần code lấy lại từ thực nghiệm cũ (mạng cổng của TN1, Transformer của TN3, phép tách VGG16 của `splitnet.py`) đã được chép vào `scripts/dcpair.py`.

## 1. Cấu hình

Chỉ cần đổi `N_IMAGES`. Mọi tham số khác đều có lý do ở bảng bên dưới. Cell này chỉ đặt hằng số và đường dẫn, không tính gì.

**Vì sao chạy CPU.** float64 trên GPU phổ thông rất chậm (RTX 3050 chạy float64 bằng 1/64 tốc độ float32), và GPU có thể đang chạy việc khác. Đo cả ba phương pháp trên cùng một CPU thì thời gian mới so được với nhau.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""          # float64 runs on the CPU (reason above); must precede `import torch`
import sys, json, time, gc, platform, subprocess, logging, importlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mpmath as mp
import torch
import torchvision
from PIL import Image
from IPython.display import display, Markdown

N_IMAGES = 1          # <- số duy nhất cần đổi: 1 = chạy thử, 5 = chạy đủ

N_FULL = 5
assert 1 <= N_IMAGES <= 50
SMOKE = N_IMAGES < N_FULL
DELTA = 1e-6
THETA, BIG_THETA, ALPHA = 0.1, 10.0, 0.4
C_PLACEHOLDER = 1.0
MP_DPS_TOY, MP_DPS_MHA = 60, 50
MLP_DIN, MLP_WIDTH, MLP_DEPTH, MLP_SEED = 8, 16, 40, 0
GATE_DEPTH, GATE_SEED, GATE_XSTAR, GATE_RHO = 40, 1, 1.0, 0.1
MHA_D, MHA_H = 6, 2
MHA_DEPTHS = [1, 2] if SMOKE else [1, 2, 3, 4, 6, 8, 12]
MHA_STEPS = 50 if SMOKE else 2000
MHA_LR, MHA_BATCH, MHA_CLIP, MHA_INIT_STD = 1e-3, 256, 1.0, 0.11
MHA_Z0_SEED, MHA_TEST_SEED, MHA_FD_H = 2026, 999, "1e-20"
REPEATS_TOY = 1 if SMOKE else 5
REPEATS_VGG = 1 if SMOKE else 3
HM_COMMIT = "31be19b460358a69516452402d41bd650845a344"

PARAMS = pd.DataFrame([
    ("N_IMAGES", N_IMAGES, "số ảnh VGG16; 1 = chạy thử (mọi cell chạy, số liệu chưa có giá trị kết luận), 5 = chạy đủ"),
    ("dtype", "float64", "yêu cầu của thực nghiệm; HM §3.2: mọi phép tính của phép tách dùng torch.float64"),
    ("thiết bị", f"CPU, {os.cpu_count()} luồng", "float64 trên GPU phổ thông chậm (RTX 3050: 1/64 tốc độ float32), GPU có thể đang bận; "
                                                  "ba phương pháp đo trên cùng thiết bị"),
    ("DELTA", DELTA, "ngưỡng của độ sâu chứng thực, như Paper1 (Hình 3) và TN1"),
    ("THETA, BIG_THETA", f"{THETA}, {BIG_THETA}", "HM §4.1; minimize_scaling_factor, minimize_threshold trong "
                                                  "configs/nb01/methods/splitcam_a40_sc.json"),
    ("chế độ co của B", "threshold_scale_both_simultaneously", "cùng file config"),
    ("ghi đè của B", "kích hoạt + gradient", "HYBRID_CONFIG của HM (xai/evaluation/unified_config.py); "
                                            "đúng lựa chọn 'cho dùng mạng gốc ghi đè'"),
    ("ALPHA", ALPHA, "adaptive_shift_independent_40 trong cùng config (các hàng SplitCAM/SplitGrad của Bảng 1)"),
    ("maxpool của A, B", "dạng lồi", "use_monotone_maxpool: true trong config"),
    ("C_PLACEHOLDER", C_PLACEHOLDER, "hệ số gauge của C ở các mạng không có cận dạng đóng; tại tâm kết quả không phụ thuộc C (§4 kiểm)"),
    ("MP_DPS_TOY, MP_DPS_MHA", f"{MP_DPS_TOY}, {MP_DPS_MHA}", "đáp án chuẩn: 60 chữ số như Paper1/TN1; 50 cho MHA như TN3 "
                                                             "(sai phân trên 23 ô tốn thời gian; vẫn dư cho sai số đo tới 1e-18)"),
    ("MLP_DIN, MLP_WIDTH, MLP_DEPTH", f"{MLP_DIN}, {MLP_WIDTH}, {MLP_DEPTH}",
     "rộng 16: cơ số tăng của χ_A ≈ 1.13·√16 ≈ 4.5 (ước lượng; §5 đo lại), nên A dự kiến sụp quanh L ≈ 24, nằm trong dải quét 40; "
     "đầu vào 8 chiều đủ nhỏ để gradient mpmath rẻ"),
    ("MLP_SEED", MLP_SEED, "seed cố định; trọng số He, rồi mỗi tầng nhân một hệ số dương để ‖a_ℓ‖ = ‖x*‖ (chuẩn hoá kiểu LSUV): "
                           "f_L giữ O(1) ở mọi độ sâu, nên sự bùng nổ đo được là của biểu diễn"),
    ("mạng cổng", f"L ≤ {GATE_DEPTH}, seed {GATE_SEED}, x* = {GATE_XSTAR}, ρ = {GATE_RHO}",
     "giữ nguyên TN1: hệ số dốc ±U(0.3, 0.7) với dấu ngẫu nhiên (f không lồi), tiền kích hoạt tại x* trong U(0.8, 1.2) "
     "(mọi ReLU hoạt động trên Ω); ρ chỉ dùng cho cận C dạng đóng"),
    ("VGG16", "IMAGENET1K_V1, lớp dự đoán", "như HM và nb01; giải thích logit của lớp dự đoán (HM, Hình 4/6)"),
    ("MHA", f"d = {MHA_D}, H = {MHA_H}, N ∈ {MHA_DEPTHS}", "giữ nguyên TN3 (ICL hồi quy tuyến tính, 6 token, MLP 2d, "
                                                         "không LayerNorm/bias); chạy thử chỉ N ∈ {1, 2}"),
    ("MHA_STEPS, MHA_LR, MHA_BATCH, MHA_CLIP, MHA_INIT_STD", f"{MHA_STEPS}, {MHA_LR}, {MHA_BATCH}, {MHA_CLIP}, {MHA_INIT_STD}",
     "như TN3 (2000 bước Adam); chạy thử 50 bước chỉ để kiểm cell"),
    ("MHA_Z0_SEED, MHA_TEST_SEED, MHA_FD_H", f"{MHA_Z0_SEED}, {MHA_TEST_SEED}, {MHA_FD_H}",
     "như TN3; bước sai phân 1e-20 với 50 chữ số: sai số cắt cụt ~1e-40, sai số làm tròn ~1e-30"),
    ("REPEATS_TOY, REPEATS_VGG", f"{REPEATS_TOY}, {REPEATS_VGG}", "thời gian lấy trung vị nhiều lần chạy để bớt nhiễu hệ điều hành; "
                                                                  "VGG16 float64 tốn thời gian nên 3 lần; chạy thử 1 lần"),
    ("HM_COMMIT", HM_COMMIT[:7], "phiên bản repo HM dùng cho phép kiểm khớp ở §7.2"),
], columns=["param", "value", "reason"])

NB_ID = "nb02"         # mã notebook: results/, models/ và tên hình, bảng đều mang mã này
EXP_DIR = Path.cwd().resolve()
assert (EXP_DIR / "pyproject.toml").exists(), "hãy mở notebook từ thư mục gauge-centering-xai/experiments"
DOCS = EXP_DIR.parents[1]
HM_REPO = Path(os.environ.get("HM_REPO", DOCS / "DC-Decomposition-for-Explanability")).resolve()
DATA_ROOT = Path(os.environ.get("DATA_ROOT", DOCS / "data")).resolve()
IMAGENET, IMAGENET_S = DATA_ROOT / "imagenet", DATA_ROOT / "imagenet-s"
RUN_DIR = EXP_DIR / "results" / NB_ID / f"n{N_IMAGES}"
# attention models: trained once and shared by every full run; the 50-step smoke models stay inside the run folder
MODEL_DIR = RUN_DIR / "models" if SMOKE else EXP_DIR / "models" / NB_ID
DATA_DIR = RUN_DIR / "data"                       # raw numbers behind every table and figure
for d in (DATA_DIR, RUN_DIR / "figures", RUN_DIR / "tables", MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(EXP_DIR / "scripts"))
import dcpair as dc                               # pair arithmetic, methods A/B/C, memory ledger
import nb02_report as rep                         # tables and figures, read only from DATA_DIR
from nb_progress import Progress                  # progress bar + progress.json for scripts/run_notebook.py
LABELS = rep.default_labels()                     # display names; edit them in §11

torch.set_num_threads(os.cpu_count())
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 120)
PARAMS.to_csv(DATA_DIR / "params.csv", index=False)
print(f"N_IMAGES = {N_IMAGES} ({'CHẠY THỬ' if SMOKE else 'CHẠY ĐỦ'}); CPU, {torch.get_num_threads()} luồng")
print(f"dữ liệu : {DATA_ROOT}\nmô hình : {MODEL_DIR}\nkết quả : {RUN_DIR}")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.show_csv(RUN_DIR, "params.csv", LABELS))

## 2. Môi trường

Cell này kiểm ba điều:
- repo HM đúng commit `31be19b` và không có thay đổi cục bộ (cần cho phép kiểm khớp ở §7.2);
- có dữ liệu ảnh;
- trọng số VGG16 đã nằm trong cache của torch (nếu chưa thì tải về).

Phiên bản các thư viện được ghi vào `data/env.json`.

In [ ]:
def git(*args):
    return subprocess.run(["git", "-C", str(HM_REPO), *args], capture_output=True, text=True, check=True).stdout.strip()

hm_commit, hm_dirty = git("rev-parse", "HEAD"), git("status", "--porcelain", "--untracked-files=no")
assert hm_commit == HM_COMMIT, f"repo HM đang ở {hm_commit}, cần {HM_COMMIT}"
assert hm_dirty == "", f"repo HM có thay đổi cục bộ:\n{hm_dirty}"
assert (IMAGENET / "val").is_dir() and (IMAGENET_S / "ImageNetS50" / "validation-segmentation").is_dir(), \
    f"thiếu ảnh trong {DATA_ROOT}: chạy scripts/prepare_data.py"
torchvision.models.VGG16_Weights.IMAGENET1K_V1.get_state_dict(progress=False)   # into the torch cache once
ENV = {"python": platform.python_version(), "platform": platform.platform(), "processor": platform.processor(),
       "cpu_threads": torch.get_num_threads(), "torch": torch.__version__, "torchvision": torchvision.__version__,
       "numpy": np.__version__, "pandas": pd.__version__, "mpmath": mp.__version__, "hm_commit": hm_commit,
       "n_images": N_IMAGES, "smoke": SMOKE}
(DATA_DIR / "env.json").write_text(json.dumps(ENV, indent=2), encoding="utf-8")
print("môi trường: đạt;", ENV["processor"] or ENV["platform"])

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.show_env(RUN_DIR, LABELS))

## 3. Lý thuyết chung

Mục này nêu các mệnh đề mà phần đo sẽ kiểm, kèm chứng minh ngắn. Ký hiệu: $u = 2^{-53}$ là đơn vị làm tròn của float64; $(P, Q)$ là cặp của $f$, vô hướng (giá trị) hoặc véc-tơ (gradient, kích hoạt của một tầng).

### 3.1 Số điều kiện $\chi$ và Bổ đề 3

$$\chi = \frac{\lVert\,|P| + |Q|\,\rVert_2}{\lVert P - Q\rVert_2} \ \ge\ 1.$$

Với $P, Q \ge 0$ vô hướng, đây chính là $\chi_{\mathrm{DC}}$ của Paper1 (Định nghĩa 2).

**Bổ đề 3 (dạng chuẩn).** Nếu $\hat P = P + e_P$ và $\hat Q = Q + e_Q$ với $|e_P| \le \eta|P|$, $|e_Q| \le \eta|Q|$ theo từng thành phần, thì
$$\frac{\lVert(\hat P - \hat Q) - (P - Q)\rVert_2}{\lVert P - Q\rVert_2} \le \eta\,\chi.$$

*Chứng minh.* Theo từng thành phần, $|e_P - e_Q| \le \eta(|P| + |Q|)$. Lấy chuẩn 2 hai vế rồi chia cho $\lVert P - Q\rVert_2$. $\square$

Ba hệ quả dùng trong notebook:
1. Trước phép trừ cuối, A chỉ cộng, nhân và lấy max các số **không âm**. Vì vậy $P$, $Q$ được tính với sai số tương đối $\eta \approx k u$, với $k$ là số phép tính trên đường dài nhất. Toàn bộ thiệt hại dồn vào phép trừ: sai số $\lesssim k u \chi$. Đường $u\chi$ được vẽ làm **dự đoán**, cạnh sai số đo được.
2. $|P| + |Q|$ là tổng các số không âm, nên $\chi$ **tính được tin cậy ngay trong float64** (sai số tương đối $\le \gamma_k$), dù $\hat P - \hat Q$ đã mất hết chữ số. Ở VGG16, nơi không có đáp án nhiều chữ số, $\chi$ theo tầng là chẩn đoán chính.
3. Khi $\chi \gtrsim 1/u \approx 9\times10^{15}$, phép trừ mất mọi chữ số và sai số tương đối xấp xỉ 1.

Trong mọi bảng, mẫu số của $\chi$ là giá trị **đúng**: $f$ tính bằng mpmath, hoặc forward float64 của mạng gốc ở VGG16. Không dùng $\hat P - \hat Q$ làm mẫu số.

### 3.2 Hai chế độ tăng trưởng (Paper1, Mệnh đề 8 và Định lý 9)

Gọi $M_\ell = \max(|P_\ell|, |Q_\ell|)$ là độ lớn thành phần sau tầng hoặc khối $\ell$.
- **Nhân với trọng số cố định** (affine, conv) là một nút mức giá trị với nhân tử bị chặn: $M_{\ell+1} \le 2M_\ell N$. Độ lớn tăng **mũ đơn**, nên $\log\chi_\ell$ tăng tuyến tính theo $\ell$. Tốc độ do $\lVert|W|\rVert$ quyết định, không phải $\lVert W\rVert$ [HM, (29)].
- **Nút tự cường** là nút nhân mà cả hai thừa số cùng biến thiên. Với luật bình phương, $M_{\ell+1} \ge \tfrac12 M_\ell^2$: độ lớn tăng **mũ kép**, và $\log\chi$ gấp đôi sau mỗi nút.

Để phân biệt hai chế độ bằng số, dùng **tỉ số tăng trưởng**
$$r_\ell = \frac{\log\chi_{\ell+1} - \log\chi_\ell}{\log\chi_\ell - \log\chi_{\ell-1}}.$$

| Mạng | Nút tự cường của A | $r$ dự đoán |
| --- | --- | --- |
| ReLU MLP | không có | $\approx 1$ |
| mạng cổng | $g_\ell \cdot h_{\ell-1}$, một nút mỗi tầng | $\approx 2$ |
| VGG16 | không có | $\approx 1$, dao động vì độ rộng đổi theo tầng |
| MHA linear | $QK^\top$ và $SV$, hai nút mỗi khối | $\approx 4$ mỗi khối |
| MHA softmax | — (A không định nghĩa) | — |

TN1 đã đo được $r = 2$ trên mạng cổng. TN3 đã đo được $\log_{10}\chi_A = 2.7,\ 10.3,\ 42,\ 158$ với $N = 1, \dots, 4$, tức $r \approx 4$.

### 3.3 Phương pháp B làm gì

Một lần chạy B gồm bốn bước: lượt đi và lượt về của **mạng gốc** (để có cache), lượt đi của cặp tách, rồi lượt về của bốn độ nhạy. Hai mệnh đề sau cho biết B chính xác nhờ đâu.

**Mệnh đề B1 (co giãn không đổi $\chi$).** $\chi(\theta P, \theta Q) = \chi(P, Q)$ với mọi $\theta > 0$. Vì vậy riêng phép co $\theta$ chỉ dời số mũ, không cứu được phép trừ (Paper1, §2.3).

**Mệnh đề B2 (co rồi ghi đè nghĩa là đặt lại hiệu bằng cache).** Gọi $(p, m)$ là cặp ngay sau một chunk, $a$ là kích hoạt đã cache của mạng gốc, và $\tau \in \{0, 1\}$ là chỉ báo co ($\tau = 1$ khi $\max(|p|, |m|) > \Theta$). Cặp mà B chuyển sang chunk tiếp theo là
$$p' = s + \tfrac a2,\qquad m' = s - \tfrac a2,\qquad s = \theta^\tau\,\tfrac{p + m}{2}.$$

*Chứng minh.* Sau khi co, cặp là $(\theta^\tau p, \theta^\tau m)$. Phép ghi đè đặt $e = a - \theta^\tau(p - m)$, rồi $p' = \theta^\tau p + e/2$ và $m' = \theta^\tau m - e/2$. Cộng và trừ hai đẳng thức này. $\square$

Hệ quả:
- $p' - m' = a$ ở **mọi** chunk. Hiệu được **chép từ mạng gốc**; hiệu mà cặp tự tính (có thể đã hỏng) bị bỏ đi. Cặp chỉ giữ phần chung $s$, đúng là phần gauge (`paper-idea.md`, §2.2).
- Giá trị cuối của B là $f$ cộng sai số làm tròn của bước ghi đè cuối, cỡ $u\,\lVert|p'| + |m'|\rVert/|f|$; sai số này không tích luỹ theo độ sâu. B đúng tới sai số máy chừng nào $s$ còn vừa phải. Nói cách khác, độ chính xác của B là độ chính xác của cache.
- Nếu $|a| \le 2|s|$ theo từng thành phần, thì $\chi' = \lVert 2s\rVert/\lVert a\rVert = \theta^\tau\lVert p + m\rVert/\lVert a\rVert$. Mỗi lần co, $\chi$ giảm $1/\theta = 10$ lần. Điều đó chỉ xảy ra nhờ cache (Mệnh đề B1).

**Điều kiện ổn định (ước lượng thô, sẽ đo).** Gọi $K$ là độ lớn của cặp sau ổn định. Một chunk biến độ lớn $K$ thành $G(K)$, rồi phép co nhân thêm $\theta$ nếu kết quả vượt $\Theta$. Phần $s$ bị chặn nếu $\theta\,G(K) \le K$ tại $K \approx \Theta$:
- với trọng số cố định, $G(K) = gK$, $g \approx \lVert|W|\rVert$: cần $\theta g \le 1$;
- với nút tự cường, $G(K) \approx cK^2$: cần $\theta c\Theta \le 1$.

Ở điểm attention $S = \sum_{i \le d_h} q_i k_i$ với luật Eq. 19, $c \approx 2d_h = 6$, nên $\theta c\Theta = 6 > 1$. Nếu kích hoạt chạm ngưỡng $\Theta$, phần $s$ lại tăng mũ kép dù có cache. Một mạng co (đã huấn luyện) có thể không bao giờ chạm ngưỡng; §8 đo độ lớn sau từng chunk để biết.

**Lượt về của B.** Code HM khởi đầu bốn độ nhạy $(pp, np, pn, nn) = (e/2, 0, 0, e/2)$. Ở đây $pp = \partial g/\partial z^+$, $np = \partial g/\partial z^-$, $pn = \partial h/\partial z^+$, $nn = \partial h/\partial z^-$, và $e$ là véc-tơ đơn vị của lớp được giải thích. Sau đó, đi từ chunk cuối về chunk đầu, mỗi chunk làm ba việc:
1. ánh xạ bốn độ nhạy về đầu vào của chunk, dùng mặt nạ ReLU và chỉ số max-pool của mạng gốc;
2. dịch: trừ $\alpha(pp + np)$ khỏi $pp$ và $np$, trừ $\alpha(pn + nn)$ khỏi $pn$ và $nn$;
3. ghi đè: cộng $\pm c$, với $c = \tfrac14\big[\partial f/\partial(\text{đầu vào}) - \big((pp - np) - (pn - nn)\big)\big]$, để $(pp - np) - (pn - nn)$ bằng gradient đã cache.

Phép dịch trừ cùng một lượng khỏi $pp$ và $np$ (và khỏi $pn$, $nn$), nên không đổi bất biến $(pp - np) - (pn - nn)$. Ở đầu vào, B trả về $\delta^g - \delta^h$, với $\delta^g = pp - np$ và $\delta^h = pn - nn$. Sau bước ghi đè cuối, đại lượng này bằng $\nabla f$ đã cache, cộng sai số cỡ $u\lVert|\delta^g| + |\delta^h|\rVert$. Như vậy gradient của B cũng được chép từ mạng gốc. Đại lượng $\chi_{\text{grad}} = \lVert|\delta^g| + |\delta^h|\rVert/\lVert\nabla f\rVert$ đo độ lớn của phần gauge mà $\alpha$ chọn.

**Ba chỗ code chính thức khác bài báo.** B ở đây làm đúng như code; các điểm này được tìm ra khi viết §7.2.
1. Chunk `x` cho ra $(x/2, -x/2)$, nhưng các tầng đọc đầu vào lại nhận $(x, 0)$.
2. $\alpha$ được dịch sau **mọi** chunk, kể cả ReLU, pool và dropout, chứ không chỉ sau tầng có trọng số như Claim 2 của bài.
3. Lượt về qua Conv2d trả hai độ nhạy giữa về chỗ của nhau ($np \leftrightarrow pn$), khác với Linear và khác với đạo hàm thật. Bất biến $(pp - np) - (pn - nn)$ đối xứng theo phép đổi chỗ này, nên phép ghi đè không phát hiện ra.
   - Giá trị và gradient mà notebook này đo không bị ảnh hưởng.
   - $\delta^g$, $\delta^h$ riêng rẽ thì bị ảnh hưởng: $\chi_{\text{grad}}$ của B ở VGG16, và có thể cả các bản đồ dùng chúng (SplitGrad, SplitCAM), điều này cần kiểm riêng.
   - §4 chứng minh bằng autograd bản nào là đạo hàm thật.

### 3.4 Hai luật nhân và tính lồi

Có hai cách viết tích $uv$ khi cả $u = P - Q$ và $v = R - S$ đều biến thiên.
- **Luật bình phương** (Paper1, Bổ đề 1), dùng cho A:
  $$uv = \tfrac12\big[(P+R)^2 + (Q+S)^2\big] - \tfrac12\big[(P+S)^2 + (Q+R)^2\big].$$
  Hai thành phần lồi, vì là bình phương của tổng các hàm lồi không âm. Nhưng $M' \ge \tfrac12 M^2$: mũ kép.
- **Luật mức giá trị** (HM Eq. 19; code `direct_mat_mul_split.py`), dùng cho B: $uv = (PR + QS) - (PS + QR)$. Ở đây $\tfrac12 MN \le M' \le 2MN$. Độ lớn tăng mũ đơn nếu một thừa số "tươi" và bị chặn (như $g_\ell$ ở mạng cổng), và tăng mũ kép nếu cả hai thừa số đều tích luỹ (điểm attention; Định lý 9).

Về tính lồi của luật mức giá trị:
- **Một biến.** Tích của hai hàm lồi, không âm, cùng không giảm là lồi, vì $(PR)'' = P''R + 2P'R' + PR'' \ge 0$. Ở mạng cổng vô hướng với $x > 0$, mọi thành phần của B đều không giảm theo $x$, nên cặp của B **lồi** và chỉ tăng mũ đơn.
- **Từ hai biến trở lên** thì không còn đúng. Với $P = x_1$, $R = x_2$ thì $PR = x_1x_2$, có Hessian $\begin{pmatrix}0 & 1\\ 1 & 0\end{pmatrix}$, không nửa xác định dương. Vì vậy ở MHA, cặp của B **không phải** một phân tách DC. Định lý 4 [HM, App. E] khẳng định ngược lại; khẳng định đó chỉ đúng với một biến.

Khi đọc kết quả mạng cổng: A sụp sớm là do luật bình phương. Mũ thứ hai là cái giá phải trả cho tính lồi trong cách dựng đệ quy, đúng như tiêu đề của Paper1.

### 3.5 Phương pháp C tại tâm

Đặt $Q(x) = \frac C2\lVert x - x^\star\rVert^2$ và $P = f + Q$. Tại $x = x^\star$, $x - x^\star = 0$ **đúng tuyệt đối** trong float64, nên $Q = 0$ và $\nabla Q = 0$ cũng đúng tuyệt đối, với mọi $C$:
$$\hat v = (f + 0) - 0 = f,\qquad \hat g = (\nabla f + 0) - 0 = \nabla f,\qquad \chi_{\text{val}} = \chi_{\text{grad}} = 1.$$

C được **tính thật** bằng lượt đi và autograd float64, không lấy từ đáp án chuẩn. Sai số của C chính là sai số của lượt đi và lan truyền ngược float64. Giá trị của $C$ không ảnh hưởng tại tâm (§4 kiểm bằng hai giá trị $C$). Notebook dùng cận chặt dạng đóng của TN1 ở mạng cổng, và $C = 1$ làm giá trị giữ chỗ ở các mạng khác.

Hai giới hạn cần nói rõ:
- Tại tâm, C trùng với lượt đi + backprop. Đóng góp của C là: với giá của backprop, không cần cache hay siêu tham số, nó cho một cặp DC **hợp lệ** trên $\Omega_\rho$ (khi $C \ge \sup_{\Omega_\rho}\lVert\nabla^2 f\rVert$), với $\chi = 1$ tại tâm.
- Ở mạng ReLU (MLP, VGG16), $f$ tuyến tính từng khúc. Trong vùng kích hoạt chứa $x^\star$, $C = 0$; còn qua một nếp gấp lõm thì không có $C$ hữu hạn nào đủ (`paper-idea.md`, §4.3, vấn đề 2). Notebook này chỉ đánh giá tại tâm; chứng thực trên vùng để notebook sau.

### 3.6 Chi phí và các dự đoán

**Bộ nhớ** là tổng số byte của mọi tensor mà phương pháp phải giữ; mỗi vùng nhớ chỉ đếm một lần (`dc.Ledger`).

| | trọng số | cache | autograd |
| --- | --- | --- | --- |
| A | $W^+$, $W^-$ | — | tensor lưu cho lượt về của cặp hai luồng |
| B | $W$, $W^+$, $W^-$ | kích hoạt và gradient của mọi chunk, mặt nạ | tensor lưu cho lượt về của mạng gốc |
| C | $W$ | — | tensor lưu cho lượt về của mạng gốc |

Ở mạng cổng và MHA, B chỉ tính giá trị: HM không định nghĩa lượt về cho nút nhân, và code chính thức chỉ để một hàm đồng nhất giữ chỗ. Vì vậy bộ nhớ của B ở hai mạng này không có phần gradient.

**Thời gian** là trung vị của nhiều lần chạy, tính cả giá trị lẫn gradient. Con số chỉ có ý nghĩa ở VGG16 và phần nào ở MHA; ở các mạng nhỏ, chi phí gọi hàm Python lấn át.

**Độ sâu chứng thực** $L^*(\delta)$ là độ sâu lớn nhất sao cho sai số dưới $\delta = 10^{-6}$ ở **mọi** độ sâu từ 1 tới đó.

**Dự đoán**, chấm tự động ở §10:

| # | Dự đoán | Quy tắc đạt | Căn cứ |
| --- | --- | --- | --- |
| D1 | Sai số của A bị chặn bởi $k u\chi_A$ | sai số giá trị của A $\le 10^3\,u\chi_A$ ở mọi độ sâu chưa tràn | Bổ đề 3; mạng dài nhất ở đây có dưới $10^3$ phép tính trên một đường (MLP: 40 tầng × 16 số hạng) |
| D2 | Mũ đơn ở MLP và VGG16 | trung vị của $r$ trong $[0.5, 1.5]$ | Định lý 9, nhân tử bị chặn |
| D3 | Mũ kép ở mạng cổng | trung vị của $r$ trong $[1.5, 2.5]$, và $L^*_A$ lệch TN1 (5 với giá trị, 4 với gradient) không quá 1 | luật bình phương; TN1 |
| D4 | Mũ kép ở MHA linear | trung vị của $r$ trong $[3, 5]$ trên các khối liên tiếp | hai nút bình phương mỗi khối; TN3 |
| D5 | B đúng ở mọi độ sâu | $L^*_B$ bằng độ sâu lớn nhất, ở mọi mạng có B | Mệnh đề B2 |
| D6 | C đúng ở mọi độ sâu | $L^*_C$ bằng độ sâu lớn nhất, và sai số $\le 10^{-12}$ | §3.5 |
| D7 | C tốn ít bộ nhớ nhất | bộ nhớ của C nhỏ hơn của A và của B, ở mọi mạng và mọi độ sâu | bảng chi phí |
| D8 | B chậm hơn C | thời gian của B lớn hơn của C ở VGG16 | B làm toàn bộ việc của C rồi thêm hai lượt tách |

## 4. Engine và kiểm tra đúng đắn

`scripts/dcpair.py` có hai engine:
- `SeqNet` cho mạng viết thành dãy chunk (MLP, VGG16); engine này có lượt về của HM.
- `Ops` cho mạng viết một lần như một hàm của "phép tính" (mạng cổng, MHA). Cùng một hàm cho ra mạng gốc (torch hoặc mpmath), cặp của A và cặp của B.

Trước khi đo, ta kiểm:

| Kiểm tra | Kỳ vọng |
| --- | --- |
| cặp A là phân tách DC đúng tuyệt đối: $P - Q = f$, tính bằng mpmath | sai lệch $\le 10^{-50}$ (MHA: $\le 10^{-40}$, vì 50 chữ số và $\chi \approx 30$) |
| A float64 (autograd) khớp mpmath ở độ sâu nông | sai số $\le 10^{-13}$ |
| C tại tâm không phụ thuộc $C$ ($C = 1$ và $C = 10^6$) | giống nhau từng bit |
| B: sau mỗi chunk, $P - Q$ bằng kích hoạt đã cache | sai số $\le 10^{-14}$ |
| lượt về tách với $\alpha = 0$, không ghi đè, dùng **đạo hàm thật** ở conv, so với autograd của cặp (CNN nhỏ) | sai lệch $\le 10^{-12}$ |
| như trên, nhưng lượt về qua conv **như code HM** | **không** khớp (lỗi đổi chỗ, §3.3) |

Cột `đạt` nghĩa là kết quả đúng như kỳ vọng. Nếu có phép kiểm không như kỳ vọng, notebook dừng ở đây.

In [ ]:
checks = []

def check(name, network, value, tol, expected=True, detail=""):
    ok = bool(value <= tol)
    checks.append(dict(check=name, network=network, value=float(value), tol=tol, expected=expected,
                       passed=ok == expected, detail=detail))

# 1. the naive pair is an exact DC decomposition (mpmath)
mp.mp.dps = MP_DPS_TOY
Wc, woc, xc = dc.mlp_make(MLP_DIN, MLP_WIDTH, 3, MLP_SEED)
rows_c = dc.mlp_truth_and_chi_mp(Wc, woc, xc)
check("A: P − Q = f đúng tuyệt đối (mpmath)", "mlp",
      max(float(abs(r["P"] - r["Q"] - r["f"]) / abs(r["f"])) for r in rows_c), 1e-50, detail="L = 1, 2, 3")
wg, bg = dc.gate_make(4, GATE_SEED, GATE_XSTAR)
Pg, Qg, _, _ = dc.gate_pair_mp(wg, bg, GATE_XSTAR)
fg = dc.gate_truth(wg, bg, GATE_XSTAR)[0]
check("A: P − Q = f đúng tuyệt đối (mpmath)", "gate", float(abs(Pg - Qg - fg) / abs(fg)), 1e-50, detail="L = 4")
mp.mp.dps = MP_DPS_MHA
Mc = dc.mha_random(1, MHA_D, MHA_H, seed=1, init_std=MHA_INIT_STD)
Zc = dc.mha_sample(1, torch.Generator().manual_seed(MHA_Z0_SEED))[0][0]
Mcm, Zcm = dc.mha_to_mp(Mc), dc.to_mp(Zc)
Pz, Qz = dc.mha_net(dc.PairOps("square"), Zcm, Mcm, "linear")
fz = dc.mha_net(dc.PlainOps(), Zcm, Mcm, "linear")
check("A: P − Q = f đúng tuyệt đối (mpmath)", "mha_linear", float(abs(Pz - Qz - fz) / abs(fz)), 1e-40,
      detail="N = 1, chưa huấn luyện")

# 2. A in float64 (autograd) agrees with mpmath at shallow depth
mp.mp.dps = MP_DPS_TOY
for L in (1, 2):
    r = dc.seq_method_A(dc.SeqNet.mlp(Wc[:L], woc), xc, 0)
    check("A float64 khớp mpmath (giá trị và gradient)", "mlp",
          max(dc.rel_err(r.value, rows_c[L - 1]["f"]), dc.rel_err_vec(r.grad, rows_c[L - 1]["grad"])), 1e-13,
          detail=f"L = {L}")

# 3. C at the centre does not depend on C
wg10, bg10 = dc.gate_make(10, GATE_SEED, GATE_XSTAR)
xg = torch.tensor(GATE_XSTAR, dtype=torch.float64)
fn_g = lambda ops, xx: dc.gate_net(ops, xx, wg10, bg10)
c1, c2 = dc.ops_method_C(fn_g, xg, [], C=1.0), dc.ops_method_C(fn_g, xg, [], C=1e6)
check("C: tại tâm kết quả không phụ thuộc C (C = 1 và 1e6)", "gate",
      abs(c1.value - c2.value) + float(np.abs(c1.grad - c2.grad).max()), 0.0, detail="L = 10, so từng bit")

# 4. B: after every chunk P - Q equals the cached activation
Wb, wob, xb = dc.mlp_make(MLP_DIN, MLP_WIDTH, 10, MLP_SEED)
rb = dc.seq_method_B(dc.SeqNet.mlp(Wb, wob), xb, 0, THETA, BIG_THETA, ALPHA)
check("B: sau mỗi chunk P − Q = kích hoạt đã cache", "mlp", max(s["err"] for s in rb.stats), 1e-14, detail="L = 10")

# 5. split backward (alpha = 0, no correction) against autograd of the split pair, on a small CNN
torch.manual_seed(0)
cnn = torch.nn.Sequential(torch.nn.Conv2d(3, 6, 3, padding=1), torch.nn.ReLU(), torch.nn.MaxPool2d(2),
                          torch.nn.Conv2d(6, 6, 3, padding=1), torch.nn.ReLU(), torch.nn.AdaptiveAvgPool2d(2),
                          torch.nn.Flatten(), torch.nn.Linear(24, 5)).double()
cnet = dc.SeqNet([dc.Identity("x")] + [dc.chunk_from_module(f"m{i}", mod) for i, mod in enumerate(cnn)])
xq = torch.randn(1, 3, 8, 8, dtype=torch.float64)
cq = cnet.original(xq)
p0, m0 = xq.clone().requires_grad_(True), torch.zeros_like(xq).requires_grad_(True)  # the official code feeds (x, 0)
p, m = p0, m0
for c in cnet.chunks[1:]:
    p, m = c.split(p, m, None)
auto = [t for out in (p[0, cq.target], m[0, cq.target]) for t in torch.autograd.grad(out, (p0, m0), retain_graph=True)]
for swap, expected in ((False, True), (True, False)):
    bw = cnet.backward_B(cq, 0.0, conv_swap=swap, correct=False)
    ours = [2 * bw.pp, 2 * bw.np, 2 * bw.pn, 2 * bw.nn]          # the official code starts from (e/2, 0, 0, e/2)
    diff = max(((a - b).norm() / b.norm().clamp_min(1e-300)).item() for a, b in zip(ours, auto))
    check("lượt về tách = autograd của cặp (α = 0, không ghi đè), " + ("conv như code HM" if swap else "đạo hàm thật"),
          "cnn", diff, 1e-12, expected=expected, detail="CNN nhỏ: conv, ReLU, maxpool, conv, ReLU, avgpool, linear")

CHECKS = pd.DataFrame(checks)
CHECKS.to_csv(DATA_DIR / "checks.csv", index=False)
print(f"{int(CHECKS['passed'].sum())}/{len(CHECKS)} phép kiểm đúng như kỳ vọng")
assert CHECKS["passed"].all(), "có phép kiểm không như kỳ vọng — xem bảng"

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.show_csv(RUN_DIR, "checks.csv", LABELS))

## 5. Mạng 1 — ReLU MLP (trọng số cố định)

$$a_0 = x,\qquad a_\ell = \mathrm{ReLU}(W_\ell a_{\ell-1}),\qquad f_L(x) = \tfrac14\,\mathbf 1^\top a_L,$$
với $x \in \mathbb R^8$, độ rộng 16, không bias, $L = 1, \dots, 40$. Mạng có độ sâu $L$ dùng $L$ tầng đầu của cùng một dãy trọng số.
- Đầu đọc có mọi trọng số dương, nên $f_L > 0$ và sai số tương đối luôn có nghĩa.
- Mỗi $W_\ell$ được nhân một hệ số dương để $\lVert a_\ell\rVert = \lVert x^\star\rVert$ tại $x^\star$. Hệ số dương không đổi dấu trọng số nào, nên không đổi cấu trúc cặp; nó chỉ giữ $f_L = O(1)$, để sự bùng nổ đo được là của **biểu diễn** chứ không phải của hàm.

Đây đúng là thiết lập của HM: trọng số cố định, biến là đầu vào, không có nút tự cường. Dự đoán:
- $\log\chi_A$ tăng tuyến tính, $r \approx 1$ (D2), và A sụp khi $u\chi_A \approx 1$. Ô 5.1 tính $\chi_A$ đúng bằng mpmath, nên độ sâu sụp được **dự đoán trước** khi đo;
- B và C đúng ở mọi độ sâu (D5, D6). Ở mạng này B có lượt về đầy đủ của HM;
- bộ nhớ của cả ba tăng tuyến tính theo $L$, và C nhỏ nhất (D7).

Đáp án chuẩn: $f_L$ và $\nabla f_L$ tính bằng mpmath 60 chữ số; gradient là tích các ma trận có mặt nạ, nên đúng tuyệt đối. $\chi_A$ đúng lấy từ chính cặp A tính bằng mpmath, lan truyền cả đạo hàm theo chiều xuôi.

In [ ]:
# 5.1 truth and exact chi of A (mpmath)
mp.mp.dps = MP_DPS_TOY
MLP_W, MLP_WOUT, MLP_X = dc.mlp_make(MLP_DIN, MLP_WIDTH, MLP_DEPTH, MLP_SEED)
t0 = time.time()
MLP_TRUTH = dc.mlp_truth_and_chi_mp(MLP_W, MLP_WOUT, MLP_X)
ex = pd.DataFrame([{"depth": r["depth"], "f_true": float(r["f"]), "log10_chi_val_A": float(mp.log10(r["chi_val"])),
                    "log10_chi_grad_A": float(mp.log10(r["chi_grad"]))} for r in MLP_TRUTH])
ex.to_csv(DATA_DIR / "mlp_exact.csv", index=False)
slope = np.polyfit(ex["depth"], ex["log10_chi_val_A"], 1)[0]
bad = ex.loc[ex["log10_chi_val_A"] + np.log10(dc.U) > np.log10(DELTA), "depth"]
print(f"đáp án và χ_A đúng (mpmath): {time.time() - t0:.1f} s; f_L ∈ [{ex.f_true.min():.2f}, {ex.f_true.max():.2f}]")
print(f"log10 χ_A tăng {slope:.2f} mỗi tầng (cơ số {10 ** slope:.2f}); "
      f"dự đoán L*_A ≈ {int(bad.min()) - 1 if len(bad) else MLP_DEPTH} (u·χ_A vượt δ từ L = {int(bad.min()) if len(bad) else '—'})")

In [ ]:
# 5.2 the three methods at every depth
rows = []
with Progress(RUN_DIR, "§5 ReLU MLP", MLP_DEPTH, unit="độ sâu") as bar:
    for tr in MLP_TRUTH:
        L = tr["depth"]
        net = dc.SeqNet.mlp(MLP_W[:L], MLP_WOUT)
        runs = {"A": lambda: dc.seq_method_A(net, MLP_X, 0),
                "B": lambda: dc.seq_method_B(net, MLP_X, 0, THETA, BIG_THETA, ALPHA),
                "C": lambda: dc.seq_method_C(net, MLP_X, 0, C_PLACEHOLDER)}
        for m, run in runs.items():
            res, t = dc.timed(run, REPEATS_TOY)
            rows.append(dc.result_row("mlp", L, m, res, tr["f"], tr["grad"], t))
        bar.update()
MLP = pd.DataFrame(rows)
MLP.to_csv(DATA_DIR / "mlp.csv", index=False)
for m, g in MLP.groupby("method"):
    print(f"{m}: L* giá trị = {dc.certified_depth(g.depth, g.err_val, DELTA)}, "
          f"L* gradient = {dc.certified_depth(g.depth, g.err_grad, DELTA)}, bộ nhớ ở L = {MLP_DEPTH}: "
          f"{rep.fmt_bytes(g.mem_total.iloc[-1])}")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.depth_table(RUN_DIR, "mlp", LABELS, depths=[1, 5, 10, 15, 20, 22, 24, 26, 30, 40]))
rep.plot_depth(RUN_DIR, "mlp", LABELS); plt.show()

## 6. Mạng 2 — mạng cổng vô hướng (Paper1, (11))

$$h_0 = 1,\qquad g_\ell(x) = \mathrm{ReLU}(w_\ell x + b_\ell),\qquad h_\ell(x) = \mathrm{ReLU}\big(g_\ell(x)\,h_{\ell-1}(x)\big),\qquad f = h_L,$$
đúng cấu hình TN1 ($x^\star = 1$, $L \le 40$). Mỗi tầng có một nút nhân mà cả hai thừa số đều phụ thuộc $x$. Trên $\Omega = [0.9, 1.1]$ mọi ReLU đều hoạt động, nên $f = \prod_\ell(w_\ell x + b_\ell)$ và đáp án có dạng đóng.

Dự đoán:
- A (luật bình phương): $\log\chi_A$ gấp đôi mỗi tầng ($r \approx 2$); $L^*_A$ là 5 với giá trị và 4 với gradient, như TN1 (D3).
- B (Eq. 19 + cache): thừa số $g_\ell$ "tươi", nên ngay cả cặp HM chưa ổn định cũng chỉ tăng mũ đơn (§3.4); thêm cache thì B đúng ở mọi độ sâu (D5). Gradient của B: không định nghĩa, vì HM không có lượt về cho nút nhân.
- C: đúng ở mọi độ sâu (D6), với $C$ là cận chặt dạng đóng của TN1 trên $\Omega$.

In [ ]:
mp.mp.dps = MP_DPS_TOY
GW, GB = dc.gate_make(GATE_DEPTH, GATE_SEED, GATE_XSTAR)
lo, hi = GATE_XSTAR - GATE_RHO, GATE_XSTAR + GATE_RHO
assert np.minimum(GW * lo + GB, GW * hi + GB).min() > 0, "một ReLU đổi trạng thái trong Ω"
XG = torch.tensor(GATE_XSTAR, dtype=torch.float64)
rows, exact = [], []
with Progress(RUN_DIR, "§6 mạng cổng", GATE_DEPTH, unit="độ sâu") as bar:
    for L in range(1, GATE_DEPTH + 1):
        w, b = GW[:L], GB[:L]
        f_t, df_t, _ = dc.gate_truth(w, b, GATE_XSTAR)
        Pm, Qm, dPm, dQm = dc.gate_pair_mp(w, b, GATE_XSTAR)
        exact.append(dict(depth=L, f_true=float(f_t), log10_chi_val_A=float(mp.log10((Pm + Qm) / abs(f_t))),
                          log10_chi_grad_A=float(mp.log10((abs(dPm) + abs(dQm)) / abs(df_t)))))
        fn = lambda ops, xx: dc.gate_net(ops, xx, w, b)
        wts = [torch.tensor(w), torch.tensor(b)]
        C_L = dc.gate_curvature_bound(w, b, GATE_XSTAR, GATE_RHO)
        runs = {"A": lambda: dc.ops_method_A(fn, XG, wts, "square"),
                "B": lambda: dc.ops_method_B(fn, XG, wts, THETA, BIG_THETA),
                "C": lambda: dc.ops_method_C(fn, XG, wts, C=C_L)}
        for m, run in runs.items():
            res, t = dc.timed(run, REPEATS_TOY)
            rows.append(dc.result_row("gate", L, m, res, f_t, [df_t], t))
        bar.update()
pd.DataFrame(exact).to_csv(DATA_DIR / "gate_exact.csv", index=False)
GATE = pd.DataFrame(rows)
GATE.to_csv(DATA_DIR / "gate.csv", index=False)
for m, g in GATE.groupby("method"):
    print(f"{m}: L* giá trị = {dc.certified_depth(g.depth, g.err_val, DELTA)}, "
          f"L* gradient = {dc.certified_depth(g.depth, g.err_grad, DELTA) if g.err_grad.notna().any() else 'không định nghĩa'}")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.depth_table(RUN_DIR, "gate", LABELS, depths=[1, 2, 3, 4, 5, 6, 7, 8, 10, 12, 16, 20, 30, 40]))
rep.plot_depth(RUN_DIR, "gate", LABELS); plt.show()

## 7. Mạng 3 — VGG16

VGG16 của torchvision (trọng số `IMAGENET1K_V1`), float64, giải thích logit của lớp dự đoán. Đây là mạng thật và đúng thiết lập của HM: 13 conv và 3 linear, trọng số cố định, không có nút tự cường. "Độ sâu" là chỉ số tầng. Mỗi chunk của code HM là một module của torchvision: tổng cộng 41 chunk, kể cả chunk đầu vào `x` và hai dropout.

Không có đáp án nhiều chữ số cho VGG16. Tham chiếu là lượt đi và autograd float64 của chính model torchvision. Lượt đi này không có phép trừ triệt tiêu đáng kể, nên sai số của nó cỡ $10^{-15}$, quá nhỏ để ảnh hưởng tới các sai số $\gg 10^{-12}$ cần đo ở A. C dùng đúng phép tính đó, nên **sai số của C ở đây không đo được**: bảng ghi C trùng với tham chiếu. Chẩn đoán chính là $\chi$ theo tầng, tính được tin cậy trong float64 (§3.1).

Dự đoán:
- $\log\chi_A$ tăng gần tuyến tính theo tầng (D2) và vượt $1/u$ ở các tầng cuối; HM, App. C.2 báo độ lớn $\sim10^{18}$ khi không ổn định;
- B đúng ở logit (D5);
- B tốn bộ nhớ nhất và chậm hơn C (D7, D8).

### 7.1 Model và ảnh

Quy tắc chọn ảnh: duyệt các lớp của ImageNet-S-50 theo thứ tự tên; ở mỗi lớp, lấy ảnh đầu tiên (theo tên file) vừa có mask vừa có trong `docs/data/imagenet/val/<lớp>/`; dừng khi đủ `N_IMAGES` ảnh. Quy tắc cố định, không phụ thuộc kết quả của notebook khác. Tiền xử lý như HM: resize 256, cắt giữa 224, chuẩn hoá.

In [ ]:
VGG_W = torchvision.models.VGG16_Weights.IMAGENET1K_V1
VGG = torchvision.models.vgg16(weights=VGG_W).eval().double()
TRANSFORM = VGG_W.transforms()
t0 = time.time()
VNET = dc.SeqNet.from_vgg(VGG)                   # chunks with W, W+ and W- (split once, like HM's model splitting)
SPLIT_S = time.time() - t0
seg = IMAGENET_S / "ImageNetS50" / "validation-segmentation"
picked = []
for wnid in sorted(p.name for p in seg.iterdir() if p.is_dir()):
    jpg = next((IMAGENET / "val" / wnid / f"{f.stem}.JPEG" for f in sorted((seg / wnid).glob("*.png"))
                if (IMAGENET / "val" / wnid / f"{f.stem}.JPEG").exists()), None)
    if jpg is not None:
        picked.append((wnid, jpg))
    if len(picked) == N_IMAGES:
        break
X_IMG, rows = [], []
with torch.no_grad():
    for k, (wnid, f) in enumerate(picked):
        x = TRANSFORM(Image.open(f).convert("RGB"))[None].double()
        logits = VGG(x)[0]
        y = int(logits.argmax())
        rows.append(dict(image=k, wnid=wnid, file=f.name, target=y, prob=float(logits.softmax(0)[y])))
        X_IMG.append(x)
VIMG = pd.DataFrame(rows)
VIMG.to_csv(DATA_DIR / "vgg_images.csv", index=False)
print(f"VGG16 float64, {len(VNET.chunks)} chunk; tách W⁺, W⁻ một lần: {SPLIT_S:.1f} s; {len(X_IMG)} ảnh")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.show_csv(RUN_DIR, "vgg_images.csv", LABELS))

### 7.2 Khớp với code HM chính thức

Trên ảnh đầu tiên, ta chạy code chính thức (`HybridDecomposer`, đúng cấu hình ở §1, float64, CPU) và so với B của `dcpair.py`, theo từng chunk:
- lượt đi: hai luồng $(p, m)$ sau ổn định;
- lượt về: bốn độ nhạy $(pp, np, pn, nn)$ sau khi dịch và ghi đè.

**Quy tắc:** khớp nếu sai lệch tương đối $\lVert\text{ta} - \text{HM}\rVert/\lVert\text{HM}\rVert \le 10^{-10}$ ở mọi chunk và mọi đại lượng. Hai bên chạy cùng một thuật toán float64, chỉ khác thứ tự cộng, nên sai lệch kỳ vọng cỡ $10^{-15}$; ngưỡng $10^{-10}$ chừa biên cho sai lệch tích luỹ qua 41 chunk.

Cột thứ hai so với B dùng đạo hàm thật ở conv (không đổi chỗ). Sai lệch $O(1)$ bắt đầu từ conv cuối cùng là dấu vết của lỗi ở §3.3.

In [ ]:
sys.path.insert(0, str(HM_REPO))
logging.disable(logging.CRITICAL)                # the HM code logs every tensor at DEBUG level
from dc_decomposition.hybrid_split.hybrid_decomposer import HybridDecomposer
from dc_decomposition.hybrid_split.layer_markings import LayerMarkingStrategy, SplitMethod
from dc_decomposition.split_util.gradient_processor import CoordinatedGradientProcessor

HM_CFG = dict(enable_grad_caching=True, enable_grad_assimilation=True, enable_activation_caching=True,
              enable_activation_assimilation=True, minimize_mode="threshold_scale_both_simultaneously",
              minimize_with_grad=False, minimize_scaling_factor=THETA, minimize_threshold=BIG_THETA,
              grad_minimize_mode=f"adaptive_shift_independent_{round(ALPHA * 100)}", split_mode="max",
              use_monotone_maxpool=True)
x0, y0 = X_IMG[0], int(VIMG["target"][0])
t0 = time.time()
try:
    hm_net = HybridDecomposer(model=VGG, input_shape=(1, 3, 224, 224), device="cpu", precision="float64",
                              marking_strategy=LayerMarkingStrategy(default_method=SplitMethod.DIRECT),
                              backprop_mode="standard", **HM_CFG).decompose()
    assert [c.layer_plans[0].node_name for c in hm_net.chunks] == VNET.names, "chunk của HM khác chunk của dcpair"
    hm_out = {}
    hooks = [mod.register_forward_hook(
        lambda _m, _i, out, i=i: hm_out.__setitem__(i, (out[0].detach().clone(), out[1].detach().clone())))
        for i, mod in enumerate(hm_net.chunk_modules)]
    gp = CoordinatedGradientProcessor(minimize_mode=HM_CFG["grad_minimize_mode"], enable_grad_caching=True,
                                      enable_grad_assimilation=True)
    gp.prepare_original_gradient_caching(VGG, hm_net.chunks, x0, getattr(hm_net, "analyzer", None))
    pos, neg = hm_net(x0)
    onehot = torch.zeros_like(pos)
    onehot[0, y0] = 1.0
    gp.compute_gradients_with_loss(lambda o, t: torch.sum(o * t), onehot)
    hm_sens = hm_net.backward_pass_with_cached_gradients(gp)
    for h in hooks:
        h.remove()
finally:
    logging.disable(logging.NOTSET)
    root = logging.getLogger()                   # the HM import sets the root logger to DEBUG: restore quiet output
    root.setLevel(logging.WARNING)
    for h in root.handlers:
        h.setLevel(logging.WARNING)
HM_S = time.time() - t0
ours = dc.seq_method_B(VNET, x0, y0, THETA, BIG_THETA, ALPHA, keep_pairs=True, keep_sens=tuple(VNET.names))
ours_true = VNET.backward_B(ours.cache, ALPHA, keep=tuple(VNET.names), conv_swap=False)
rel = lambda a, b: ((a - b).norm() / b.norm().clamp_min(1e-300)).item()
rows = []
for i, name in enumerate(VNET.names):
    for j, q in enumerate(("fwd_pos", "fwd_neg")):
        d = rel(ours.pairs[name][j], hm_out[i][j])
        rows.append(dict(node=name, quantity=q, rel_diff_official=d, rel_diff_true=d, scale=hm_out[i][j].norm().item()))
    for j, q in enumerate(("d_pp", "d_np", "d_pn", "d_nn")):
        b_ = hm_sens[name][j]
        rows.append(dict(node=name, quantity=q, rel_diff_official=rel(ours.sens[name][j], b_),
                         rel_diff_true=rel(ours_true.kept[name][j], b_), scale=b_.norm().item()))
FID = pd.DataFrame(rows)
FID["passed"] = FID["rel_diff_official"] <= 1e-10
FID.to_csv(DATA_DIR / "hm_fidelity.csv", index=False)
print(f"code HM chạy {HM_S:.0f} s; B của dcpair khớp ở {int(FID.groupby('node')['passed'].all().sum())}/{len(VNET.names)} "
      f"chunk, sai lệch lớn nhất {FID['rel_diff_official'].max():.1e}")
del hm_net, gp, hm_out, hm_sens, pos, neg, ours, ours_true
gc.collect();

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
tab, first_conv = rep.fidelity_table(RUN_DIR, LABELS)
display(tab)
fid = pd.read_csv(DATA_DIR / "hm_fidelity.csv")
display(Markdown(f"**Kết luận: {'KHỚP' if fid['passed'].all() else 'KHÔNG KHỚP'}** (ngưỡng $10^{{-10}}$). "
                 + (f"Với đạo hàm thật ở conv, sai lệch xuất hiện từ chunk `{first_conv}` trở về đầu vào." if first_conv else "")))

### 7.3 Ba phương pháp: logit, theo tầng, chi phí

Với mỗi ảnh:
- **tham chiếu:** logit và gradient theo ảnh vào, từ lượt đi + autograd float64 của torchvision;
- **A, B, C:** giá trị, gradient, bộ nhớ, thời gian (trung vị của `REPEATS_VGG` lần chạy, không tính phần đo theo tầng);
- **theo tầng:** $\chi$, sai số bất biến $\lVert(a^+ - a^-) - a\rVert/\lVert a\rVert$ và độ lớn của cặp, cho A và B. Với B, đo thêm trước khi sửa (ngay sau chunk, trước khi co và ghi đè); đó là sai số làm tròn mà riêng chunk đó tạo ra.

In [ ]:
KIND = {c.name: dc.chunk_kind(c) for c in VNET.chunks}
N_WEIGHTED = sum(KIND[n] in ("conv", "linear") for n in VNET.names)
rows_f, rows_l = [], []
with Progress(RUN_DIR, "§7.3 VGG16", len(X_IMG), unit="ảnh") as bar:
    for k, x in enumerate(X_IMG):
        y = int(VIMG["target"][k])
        xr = x.clone().requires_grad_(True)
        f_ref = VGG(xr)[0, y]
        g_ref, = torch.autograd.grad(f_ref, xr)
        f_ref, g_ref = f_ref.item(), g_ref.flatten().numpy()
        runs = {"A": lambda: dc.seq_method_A(VNET, x, y),
                "B": lambda: dc.seq_method_B(VNET, x, y, THETA, BIG_THETA, ALPHA, measure=False),
                "C": lambda: dc.seq_method_C(VNET, x, y, C_PLACEHOLDER)}
        for m, run in runs.items():
            res, t = dc.timed(run, REPEATS_VGG)
            rows_f.append({**dc.result_row("vgg16", N_WEIGHTED, m, res, f_ref, g_ref, t), "image": k})
            del res
            gc.collect()
        # per-layer statistics: one extra run each, not timed (the reference states are for measuring only)
        ref = VNET.original(x, y, all_grads=False)
        for m, stats in (("A", dc.seq_method_A(VNET, x, y, ref_states=ref.a).stats),
                         ("B", dc.seq_method_B(VNET, x, y, THETA, BIG_THETA, ALPHA).stats)):
            rows_l += [{"image": k, "method": m, "kind": KIND[s["name"]], **s} for s in stats]
        del ref
        gc.collect()
        bar.update()
VF, VL = pd.DataFrame(rows_f), pd.DataFrame(rows_l)
VF.to_csv(DATA_DIR / "vgg_final.csv", index=False)
VL.to_csv(DATA_DIR / "vgg_layers.csv", index=False)
print(VF.groupby("method")[["err_val", "err_grad", "chi_val", "mem_total", "time_s"]].median().to_string())

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.vgg_table(RUN_DIR, LABELS))
rep.plot_vgg(RUN_DIR, LABELS); plt.show()

## 8. Mạng 4 — MHA linear (đã huấn luyện)

Transformer của TN3. Bài toán là hồi quy tuyến tính trong ngữ cảnh: 5 ví dụ $(x_i, y_i)$ với $x \in \mathbb R^3$, cộng một truy vấn. Mô hình gồm nhúng tuyến tính, $N$ khối [attention nhiều đầu + residual, MLP ReLU + residual], và đầu đọc của token truy vấn; không LayerNorm, không bias:
$$S_h = c\,Q_hK_h^\top,\qquad O_h = S_hV_h,\qquad X \leftarrow X + [O_1\cdots O_H]W_O,\qquad X \leftarrow X + \mathrm{ReLU}(XW_1)W_2.$$
Với linear attention, $c = 1/(\sqrt{d_h}\,n)$; với softmax theo hàng (§9), $c = 1/\sqrt{d_h}$. Hàm được xét theo prompt $Z$ (23 ô, bỏ nhãn ẩn của truy vấn), tại một prompt cố định $Z_0$.

Dự đoán:
- A (luật bình phương ở $QK^\top$ và $SV$): $\log\chi_A$ nhân khoảng 4 mỗi khối (D4), A sụp ở $N \approx 3$.
- B dùng Eq. 19. Cả hai thừa số đều tích luỹ, nên cặp HM chưa ổn định vẫn tăng mũ kép. B chỉ đúng nếu điều kiện ổn định ở §3.3 giữ được; ta đo độ lớn sau từng chunk để kiểm. Cặp của B không phải phân tách DC (§3.4).
- C đúng (D6).

### 8.1 Huấn luyện hoặc nạp mô hình

Mỗi cấu hình (linear và softmax, mỗi $N$) huấn luyện một lần theo đúng TN3: Adam, MSE, cắt gradient, seed khởi tạo $N$, seed dữ liệu $100 + N$. Mô hình lưu vào `models/nb02/` (lần chạy thử: vào thư mục chạy), các lần chạy sau nạp lại. Tên file ghi số bước, và cấu hình được kiểm khi nạp. MSE kiểm tra (trên 4096 prompt) được so với MSE của dự đoán hằng 0.

In [ ]:
mp.mp.dps = MP_DPS_MHA
Z_TEST, Y_TEST = dc.mha_sample(4096, torch.Generator().manual_seed(MHA_TEST_SEED))
BASELINE = float((Y_TEST ** 2).mean())
MHA_CFG = dict(d=MHA_D, H=MHA_H, steps=MHA_STEPS, lr=MHA_LR, batch=MHA_BATCH, clip=MHA_CLIP,
               init_std=MHA_INIT_STD, test_seed=MHA_TEST_SEED)
MODELS, rows = {}, []
todo = [(attn, N) for attn in ("linear", "softmax") for N in MHA_DEPTHS]
with Progress(RUN_DIR, "§8.1 mô hình MHA", len(todo), unit="mô hình") as bar:
    for attn, N in todo:
        path = MODEL_DIR / f"mha_{attn}_N{N}_steps{MHA_STEPS}.pt"
        cfg = {**MHA_CFG, "attn": attn, "N": N}
        t0 = time.time()
        if path.exists():
            ck = torch.load(path, weights_only=False)
            assert ck["config"] == cfg, f"{path.name}: cấu hình khác — xoá file để huấn luyện lại"
            source = "nạp"
        else:
            M, mse, skips = dc.mha_train(N, MHA_D, MHA_H, attn, MHA_STEPS, MHA_LR, MHA_BATCH, MHA_CLIP,
                                         MHA_INIT_STD, Z_TEST, Y_TEST)
            ck = dict(M=M, mse=mse, skips=skips, config=cfg)
            torch.save(ck, path)
            source = "huấn luyện"
        MODELS[(attn, N)] = dc.mha_heads_to_lists(ck["M"])
        rows.append(dict(attn=attn, depth=N, steps=MHA_STEPS, test_mse=ck["mse"], baseline_mse=BASELINE,
                         skipped=ck["skips"], seconds=time.time() - t0, source=source))
        bar.update()
pd.DataFrame(rows).to_csv(DATA_DIR / "mha_train.csv", index=False)
print(f"{len(MODELS)} mô hình; MSE của dự đoán hằng 0 = {BASELINE:.3f}")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.show_csv(RUN_DIR, "mha_train.csv", LABELS).round(4))

### 8.2 Ba phương pháp theo số khối

Đáp án chuẩn: $f(Z_0)$ bằng lượt đi mpmath 50 chữ số; gradient bằng sai phân trung tâm trên 23 ô ($h = 10^{-20}$). $\chi_A$ đúng lấy từ cặp A tính bằng mpmath. Với B, ta lưu độ lớn của cặp trước và sau ổn định ở mọi chunk.

In [ ]:
mp.mp.dps = MP_DPS_MHA
Z0 = dc.mha_sample(1, torch.Generator().manual_seed(MHA_Z0_SEED))[0][0]
MASK = torch.ones_like(Z0, dtype=torch.bool)
MASK[-1, -1] = False                             # the hidden query label is not an input
Z0_MP, FD_H = dc.to_mp(Z0), mp.mpf(MHA_FD_H)
rows, exact, chunks = [], [], []
with Progress(RUN_DIR, "§8.2 MHA linear", len(MHA_DEPTHS), unit="độ sâu") as bar:
    for N in MHA_DEPTHS:
        M = MODELS[("linear", N)]
        Mm = dc.mha_to_mp(M)
        f_t, g_t = dc.mha_truth(Mm, Z0_MP, "linear", MASK.numpy(), FD_H)
        Pm, Qm = dc.mha_net(dc.PairOps("square"), Z0_MP, Mm, "linear")
        exact.append(dict(depth=N, f_true=float(f_t), log10_chi_val_A=float(mp.log10((Pm + Qm) / abs(f_t))),
                          log10_chi_grad_A=float("nan")))
        fn = lambda ops, z: dc.mha_net(ops, z, M, "linear")
        wts = dc.mha_weights(M)
        runs = {"A": lambda: dc.ops_method_A(fn, Z0, wts, "square", mask=MASK),
                "B": lambda: dc.ops_method_B(fn, Z0, wts, THETA, BIG_THETA),
                "C": lambda: dc.ops_method_C(fn, Z0, wts, C_PLACEHOLDER, mask=MASK)}
        for m, run in runs.items():
            res, t = dc.timed(run, REPEATS_TOY)
            rows.append(dc.result_row("mha_linear", N, m, res, f_t, g_t, t))
            if m == "B":
                chunks += [dict(depth=N, idx=i, **s) for i, s in enumerate(res.trace)]
        bar.update()
pd.DataFrame(exact).to_csv(DATA_DIR / "mha_linear_exact.csv", index=False)
pd.DataFrame(chunks).to_csv(DATA_DIR / "mha_linear_chunks.csv", index=False)
MHL = pd.DataFrame(rows)
MHL.to_csv(DATA_DIR / "mha_linear.csv", index=False)
for m, g in MHL.groupby("method"):
    print(f"{m}: L* giá trị = {dc.certified_depth(g.depth, g.err_val, DELTA)}, "
          f"L* gradient = {dc.certified_depth(g.depth, g.err_grad, DELTA) if g.err_grad.notna().any() else 'không định nghĩa'}")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.depth_table(RUN_DIR, "mha_linear", LABELS))
rep.plot_depth(RUN_DIR, "mha_linear", LABELS); plt.show()
rep.plot_mha_chunks(RUN_DIR, LABELS, theta_big=BIG_THETA); plt.show()

## 9. Mạng 5 — MHA softmax (đã huấn luyện)

Cùng kiến trúc, với softmax theo hàng. Không có cách dựng DC đệ quy chính xác cho softmax:
- Paper1 không dựng (§2.4);
- code HM chỉ có một bản xấp xỉ bằng mạng ReLU để giữ chỗ, trong đó phép chia vẫn là phép chia thật (`approximations.py`, `SoftmaxReLUApproximation`).

Vì vậy ở mạng này **chỉ có C**: C không cần dựng gì, chỉ cần $f$ và autograd. A và B được ghi là không định nghĩa.

Dự đoán: C đúng tới sai số máy ở mọi $N$ (D6). Chứng thực trên vùng cần cận Hessian chặt chẽ và vướng vấn đề ReLU, như TN3 đã chỉ ra; phần đó để notebook sau.

In [ ]:
mp.mp.dps = MP_DPS_MHA
rows = []
with Progress(RUN_DIR, "§9 MHA softmax", len(MHA_DEPTHS), unit="độ sâu") as bar:
    for N in MHA_DEPTHS:
        M = MODELS[("softmax", N)]
        f_t, g_t = dc.mha_truth(dc.mha_to_mp(M), Z0_MP, "softmax", MASK.numpy(), FD_H)
        fn = lambda ops, z: dc.mha_net(ops, z, M, "softmax")
        res, t = dc.timed(lambda: dc.ops_method_C(fn, Z0, dc.mha_weights(M), C_PLACEHOLDER, mask=MASK), REPEATS_TOY)
        rows.append(dc.result_row("mha_softmax", N, "C", res, f_t, g_t, t))
        bar.update()
MHS = pd.DataFrame(rows)
MHS.to_csv(DATA_DIR / "mha_softmax.csv", index=False)
print(f"C: sai số giá trị lớn nhất {MHS.err_val.max():.1e}, gradient {MHS.err_grad.max():.1e}")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.depth_table(RUN_DIR, "mha_softmax", LABELS))
rep.plot_depth(RUN_DIR, "mha_softmax", LABELS); plt.show()

## 10. Tổng hợp và kiểm tra dự đoán

Bảng tổng hợp có một dòng cho mỗi cặp (mạng, phương pháp), gồm:
- độ sâu chứng thực $L^*$ của giá trị và của gradient;
- sai số lớn nhất;
- bộ nhớ và thời gian ở độ sâu lớn nhất;
- có cần cache của mạng gốc không, và các siêu tham số.

Với VGG16, $L^*$ tính theo sai số bất biến (trung vị qua các ảnh) ở 16 tầng có trọng số.

Các dự đoán D1–D8 của §3.6 được chấm đúng theo quy tắc đã ghi. Dự đoán nào thiếu số liệu thì ghi "không chấm được"; ví dụ D4 khi chạy thử, vì khi đó chỉ có $N \in \{1, 2\}$. Kết luận ghi vào `nb02_verdict.json` và `nb02_verdict.md` ở thư mục chạy.

In [ ]:
def load(name):
    p = DATA_DIR / f"{name}.csv"
    return pd.read_csv(p) if p.exists() else None

def worst(s):
    s = pd.Series(s, dtype=float)
    return np.nan if s.isna().all() else (np.inf if s.isna().any() else float(s.max()))

def r_median(logchi):
    d = np.diff(np.asarray(logchi, dtype=float))
    rs = [b / a for a, b in zip(d[:-1], d[1:]) if a > 0 and b > 0]
    return float(np.median(rs)) if rs else np.nan

HYPER = {"A": "—", "B": f"θ = {THETA}, Θ = {BIG_THETA}, α = {ALPHA}", "C": "C (không ảnh hưởng tại tâm)"}
summary = []
for net in ("mlp", "gate", "mha_linear", "mha_softmax"):
    t = load(net)
    for m in ("A", "B", "C"):
        g = t[t.method == m].sort_values("depth")
        if g.empty:
            summary.append(dict(network=net, method=m, max_depth=int(t.depth.max()), certified_val=np.nan,
                                certified_grad=np.nan, max_err_val=np.nan, max_err_grad=np.nan, mem_at_max=np.nan,
                                time_at_max=np.nan, needs_cache=m == "B", hyperparams="không định nghĩa"))
            continue
        summary.append(dict(network=net, method=m, max_depth=int(g.depth.max()),
                            certified_val=dc.certified_depth(g.depth, g.err_val, DELTA),
                            certified_grad=dc.certified_depth(g.depth, g.err_grad, DELTA) if g.err_grad.notna().any() else np.nan,
                            max_err_val=worst(g.err_val), max_err_grad=worst(g.err_grad),
                            mem_at_max=float(g.mem_total.iloc[-1]), time_at_max=float(g.time_s.iloc[-1]),
                            needs_cache=m == "B", hyperparams=HYPER[m]))
vf, vl = load("vgg_final"), load("vgg_layers")
vw = vl[vl.kind.isin(["conv", "linear"])]
for m in ("A", "B", "C"):
    f = vf[vf.method == m]
    if m == "C":
        cert = N_WEIGHTED                          # C is the reference computation itself
    else:
        med = vw[vw.method == m].groupby("state")["err"].median().sort_index()
        cert = dc.certified_depth(range(1, len(med) + 1), med.to_numpy(), DELTA)
    summary.append(dict(network="vgg16", method=m, max_depth=N_WEIGHTED, certified_val=cert, certified_grad=np.nan,
                        max_err_val=worst(f.err_val), max_err_grad=worst(f.err_grad), mem_at_max=float(f.mem_total.median()),
                        time_at_max=float(f.time_s.median()), needs_cache=m == "B", hyperparams=HYPER[m]))
SUM = pd.DataFrame(summary)
SUM.to_csv(DATA_DIR / "summary.csv", index=False)

preds = []
def pred(pid, claim, network, measured, rule, ok):
    status = "không chấm được" if ok is None else ("đạt" if ok else "không đạt")
    preds.append(dict(id=pid, claim=claim, network=network, measured=measured, rule=rule, status=status))

for net in ("mlp", "gate", "mha_linear"):
    t, ex = load(net), load(f"{net}_exact")
    a = t[(t.method == "A") & ~t.overflow.astype(bool)].merge(ex[["depth", "log10_chi_val_A"]], on="depth")
    ratio = float((a.err_val / (dc.U * 10.0 ** a.log10_chi_val_A)).max())
    pred("D1", "sai số A ≤ k·u·χ_A", net, f"max sai số/(u·χ_A) = {ratio:.2g}", "≤ 1e3", bool(ratio <= 1e3))
a = vf[vf.method == "A"]
ratio = float((a.err_val / (dc.U * a.chi_val)).max())
pred("D1", "sai số A ≤ k·u·χ_A", "vgg16", f"max sai số/(u·χ_A) = {ratio:.2g}", "≤ 1e3", bool(ratio <= 1e3))

r = r_median(load("mlp_exact").log10_chi_val_A)
pred("D2", "mũ đơn (r ≈ 1)", "mlp", f"r trung vị = {r:.2f}", "0.5 ≤ r ≤ 1.5", None if np.isnan(r) else bool(0.5 <= r <= 1.5))
chi_w = vw[vw.method == "A"].groupby("state")["chi"].median().sort_index()
r = r_median(np.log10(chi_w.to_numpy()))
pred("D2", "mũ đơn (r ≈ 1)", "vgg16", f"r trung vị = {r:.2f}", "0.5 ≤ r ≤ 1.5", None if np.isnan(r) else bool(0.5 <= r <= 1.5))

r = r_median(load("gate_exact").log10_chi_val_A)
ga = load("gate"); ga = ga[ga.method == "A"]
Lv, Lg = dc.certified_depth(ga.depth, ga.err_val, DELTA), dc.certified_depth(ga.depth, ga.err_grad, DELTA)
pred("D3", "mũ kép (r ≈ 2); L*_A như TN1", "gate", f"r trung vị = {r:.2f}; L*_A = {Lv} (giá trị), {Lg} (gradient)",
     "1.5 ≤ r ≤ 2.5; |L*_A − 5| ≤ 1, |L*_A − 4| ≤ 1", bool(1.5 <= r <= 2.5 and abs(Lv - 5) <= 1 and abs(Lg - 4) <= 1))

exm = load("mha_linear_exact").set_index("depth")
consec = [N for N in (1, 2, 3, 4) if N in exm.index]
r = r_median(exm.loc[consec, "log10_chi_val_A"]) if len(consec) >= 3 else np.nan
pred("D4", "mũ kép (r ≈ 4 mỗi khối)", "mha_linear", f"r trung vị = {r:.2f} trên N = {consec}" if not np.isnan(r) else
     f"chỉ có N = {consec}", "3 ≤ r ≤ 5", None if np.isnan(r) else bool(3 <= r <= 5))

for net in ("mlp", "gate", "mha_linear", "vgg16"):
    s = SUM[(SUM.network == net) & (SUM.method == "B")].iloc[0]
    pred("D5", "B đúng ở mọi độ sâu", net, f"L*_B = {s.certified_val:.0f}/{s.max_depth}, sai số lớn nhất {s.max_err_val:.1e}",
         "L*_B = độ sâu lớn nhất", bool(s.certified_val == s.max_depth and s.max_err_val <= DELTA))
for net in ("mlp", "gate", "mha_linear", "mha_softmax", "vgg16"):
    s = SUM[(SUM.network == net) & (SUM.method == "C")].iloc[0]
    pred("D6", "C đúng ở mọi độ sâu", net, f"L*_C = {s.certified_val:.0f}/{s.max_depth}, sai số lớn nhất {s.max_err_val:.1e}"
         + (" (C trùng tham chiếu)" if net == "vgg16" else ""),
         "L*_C = độ sâu lớn nhất, sai số ≤ 1e-12", bool(s.certified_val == s.max_depth and s.max_err_val <= 1e-12))

for net in ("mlp", "gate", "mha_linear", "vgg16"):
    t = load(net) if net != "vgg16" else vf.assign(depth=vf["image"])
    w = t.pivot_table(index="depth", columns="method", values="mem_total")
    ok = bool(((w["C"] < w["A"]) & (w["C"] < w["B"])).all())
    pred("D7", "C tốn ít bộ nhớ nhất", net, f"C/A = {float((w['C'] / w['A']).median()):.2f}, C/B = {float((w['C'] / w['B']).median()):.2f} (trung vị)",
         "C < A và C < B ở mọi độ sâu", ok)
tm = vf.groupby("method")["time_s"].median()
pred("D8", "B chậm hơn C", "vgg16", f"B: {tm['B']:.2f} s, C: {tm['C']:.2f} s, B/C = {tm['B'] / tm['C']:.1f}", "B > C", bool(tm["B"] > tm["C"]))

PRED = pd.DataFrame(preds)
PRED.to_csv(DATA_DIR / "predictions.csv", index=False)
scored = PRED[PRED.status != "không chấm được"]
label = f"{int((scored.status == 'đạt').sum())}/{len(scored)} dự đoán đạt" + (" (chạy thử: chưa có giá trị kết luận)" if SMOKE else "")
verdict = dict(label=label, n_images=N_IMAGES, smoke=SMOKE,
               failed=scored[scored.status == "không đạt"][["id", "network"]].to_dict("records"),
               not_scored=PRED[PRED.status == "không chấm được"][["id", "network"]].to_dict("records"))
(RUN_DIR / f"{NB_ID}_verdict.json").write_text(json.dumps(verdict, ensure_ascii=False, indent=2), encoding="utf-8")
lines = [f"# nb02 — kết luận: {label}", "", "| # | mạng | đo được | kết luận |", "| --- | --- | --- | --- |"]
lines += [f"| {p.id} | {LABELS['networks'].get(p.network, p.network)} | {p.measured} | {p.status} |" for p in PRED.itertuples()]
(RUN_DIR / f"{NB_ID}_verdict.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Kết luận:", label)

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(Markdown(f"### Kết luận: **{json.loads((RUN_DIR / f'{NB_ID}_verdict.json').read_text(encoding='utf-8'))['label']}**"))
display(rep.summary_table(RUN_DIR, LABELS))
display(rep.predictions_table(RUN_DIR, LABELS))
rep.plot_summary(RUN_DIR, LABELS); plt.show()

## 11. Đổi nhãn và trình bày lại

Mọi cell hiển thị chỉ đọc `data/`, và lấy tên phương pháp, tên mạng, tên cột, tiêu đề hình từ `LABELS`. Để đổi nhãn mà không tính lại:
1. nếu kernel vừa khởi động lại, chạy §1 (chỉ đặt đường dẫn, không tính gì);
2. sửa `LABELS` ở cell dưới rồi chạy nó;
3. chạy lại cell hiển thị cần thiết, hoặc cell cuối cùng để vẽ lại tất cả. Hình được ghi đè vào `figures/`, bảng vào `tables/`.

Danh sách các khoá sửa được: `rep.default_labels()`.

| File trong `data/` | Nội dung |
| --- | --- |
| `params.csv`, `env.json` | tham số và môi trường |
| `checks.csv` | các phép kiểm đúng đắn (§4) |
| `mlp.csv`, `mlp_exact.csv` | ReLU MLP: kết quả theo độ sâu; $f$ và $\chi_A$ đúng (mpmath) |
| `gate.csv`, `gate_exact.csv` | mạng cổng: như trên |
| `vgg_images.csv`, `hm_fidelity.csv` | ảnh VGG16; phép kiểm khớp với code HM |
| `vgg_final.csv`, `vgg_layers.csv` | VGG16: logit và gradient theo ảnh; thống kê theo tầng |
| `mha_train.csv` | huấn luyện hoặc nạp các mô hình attention |
| `mha_linear.csv`, `mha_linear_exact.csv`, `mha_linear_chunks.csv` | MHA linear: kết quả theo số khối; $\chi_A$ đúng; độ lớn cặp của B sau từng chunk |
| `mha_softmax.csv` | MHA softmax: C theo số khối |
| `summary.csv`, `predictions.csv` | tổng hợp và chấm dự đoán (kết luận: `nb02_verdict.json`, `nb02_verdict.md`) |

In [ ]:
importlib.reload(rep)
LABELS = rep.default_labels()
# ví dụ đổi nhãn:
# LABELS["methods"]["B"] = "B · Hidden Monotonicity"
# LABELS["networks"]["gate"] = "Gated net"
# LABELS["columns"]["err_val"] = "rel. error (value)"

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
out = rep.redraw_all(RUN_DIR, LABELS)
print("đã vẽ lại:", ", ".join(out))
display(out["summary"])